# Grains segmentation of an orientation map (3/3)

The segmentation code is in the module `LaueTools/orientationclustering.py` (`OC` below).

- two UB matrices are linked if they belong to the **same or neighbouring map points** and their **misorientation < threshold**; clusters (grains) are the connected components
- all grains of the `.fit` files are used (`_g0`, `_g1`, ... several grains per map point)
- UB matrices are orthonormalised (polar decomposition) and misorientations take into account the **crystal symmetry** (24 equivalent orientations for cubic)
- map convention: `mapdimension = invmapdims = (nb rows = slow axis, nb cols = fast axis)`, image index = row * nb cols + col
- cluster ids are sorted by decreasing size (cluster 0 is the largest)

**This notebook is part of the LaueTools package** (https://github.com/BM32ESRF/lauetools)

Author: J.-S. Micha, Last revision: Sept 2026. Tested with python 3.12 (jupyterlab, jupyterhub, VS Code)

| notebook | input | output |
|---|---|---|
| `1_index_refine.ipynb` | `.cor` files (peak search results) | `.fit` files (indexed spots, UB matrix, strain per grain) |
| `2_postprocess_maps.ipynb` | `.fit` files | maps & statistics: nb of spots, pixel deviation, strain, lattice parameters, orientation |
| `3_segmentation_grains.ipynb` | `.fit` files | grains (clusters of orientation), KAM, GROD, boundaries |

**All settings of an experiment are in a YAML configuration file**: copy `config_template.yaml` to `configs/my_experiment.yaml`, edit it and set `CONFIG` in the first code cell. The same file is used by the 3 notebooks. `configs/` contains examples of BM32 datasets (access to ESRF `/data` needed).

# SETTINGS

In [ ]:
CONFIG = 'configs/<proposal>_MgO.yaml'   # configuration file of your experiment
CONFIG = 'configs/<proposal>_MgO_2grains.yaml'   # configuration file of your experiment
CONFIG = '/data/visitor/<proposal>/bm32/<date>/PROCESSED_DATA/<sample>/<dataset>/scan0001/configs/indexing.yaml'


FIT_FOLDER = None       # None: output fit_folder of the configuration file
THRESHOLD = .8          # deg, max misorientation between neighbouring points of a grain
MIN_CLUSTER_SIZE = 15    # min nb of points of a grain

# IMPORTS

In [ ]:
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import ipympl  # interactive plots (zoom, hover) in JupyterLab / VS Code
    %matplotlib widget
except ImportError:
    %matplotlib inline

import LaueTools.indexing_batch as IB
import LaueTools.orientationclustering as OC

# LOAD CONFIGURATION

In [ ]:
cfg = IB.load_config(CONFIG)
IB.print_config(cfg)
scan, post = cfg['scan'], cfg['postprocess']
mapdims, invmapdims = scan['mapdims'], scan['invmapdims']   # (fast, slow), (slow, fast)
fit_folder = Path(FIT_FOLDER) if FIT_FOLDER else cfg['output']['fit_folder']

print('.fit files in', fit_folder)

# RUN segmentation

In [ ]:
t0 = time.time()
result, analyzer = OC.analyze_ub_matrices(input_dir=fit_folder,
                                          threshold=THRESHOLD,          # max misorientation (deg) between neighbouring points
                                          mapdimension=invmapdims,      # (nb rows = slow axis, nb cols = fast axis)
                                          prefix=scan['prefix'],        # e.g. eiger4m_0123_g0.fit
                                          symmetry=post['symmetry'],    # 'auto': cubic if all materials are cubic, None: raw angles
                                          spatial_connectivity=True,
                                          connectivity_type='8',        # '4': edges only, '8': edges + corners
                                          min_cluster_size=MIN_CLUSTER_SIZE,
                                          check_symmetry=True,          # ALARM if matrices are equivalent by symmetry but not reduced
                                          nbfiles_per_folder=IB.fitfiles_layout(cfg)['fit_nbfiles_per_folder'],   # .fit files in subfolders
                                          subfolder_prefix=scan['subfolder_prefix'])
print(f'elapsed time {time.time() - t0:.2f} s')
print(f"Grain indices: {analyzer.get_all_grain_indices()}")
OC.print_cluster_summary(result, analyzer, nb_max=20)

## symmetry check

`check_symmetry_equivalents()` compares matrices of the same and neighbouring map points. It warns when two matrices look very different (raw angle > `angle_tol`) but are equivalent by crystal symmetry, i.e. when the matrices of the .fit files have **not** been reduced to a common representative. The clustering above is symmetry-aware, so it is not affected, but any raw element-wise comparison (e.g. `GT.getRotationAngleFrom2Matrices`, maps of UB elements or Euler angles) is.

`align_to_cluster_reference()` returns the UB matrices transformed (UB.S) to the symmetry variant closest to the reference matrix of their cluster.

In [ ]:
report = OC.check_symmetry_equivalents(analyzer, symmetry=result.symmetry, angle_tol=2.0, verbose=False)
print(f"{len(report['sym_equivalent_pairs'])} neighbouring pairs equivalent by symmetry only")
for i, j, raw_angle, reduced_angle in report['sym_equivalent_pairs'][:10]:
    print(f"  {analyzer.matrices[i]} <-> {analyzer.matrices[j]}: raw {raw_angle:.1f}°, reduced {reduced_angle:.2f}°")

# UB matrices brought to a common symmetry representative within each cluster, shape (nb matrices, 3, 3)
UB_aligned = OC.align_to_cluster_reference(result, analyzer)

# CLUSTERS statistics

In [ ]:
# find cluster seen in a given sample pt (or image index)

# ClusterStats by cluster id (do not index the list returned by get_cluster_stats() with a cluster id)
stats_by_id = result.get_cluster_stats_dict(analyzer)

testimageindex = post['reference_image_index'] or cfg['run']['test_image_index']
for stats in result.get_cluster_stats_by_image(analyzer, image_index=testimageindex):
    print(f"Cluster {stats.cluster_id} contains image {testimageindex}")
    print(f"  {stats}")
    print(f"  reference UB matrix (image {stats.reference_image_index}): {stats.reference_matrix.tolist()}")
    print(f"  distribution of misorientations to reference: {OC.get_angular_deviation_distribution(stats)}")

In [ ]:
# reference UB matrix of each cluster (as written in the .fit file of the point closest to the cluster barycenter)
# printed as copy-paste ready lists (e.g. ub_matrices of the configuration file); which='mean': mean orientation
refs = OC.get_reference_matrices(result, analyzer, min_size=30, which='reference')
UBlist = [np.array(m) for m in refs.values()]

In [ ]:
# clusters sorted by orientation spread
for stats in result.get_cluster_stats_sorted(analyzer, sort_by='max_angular_deviation')[:15]:
    row, col = stats.mean_position
    print(f"Cluster {stats.cluster_id:3d}: {stats.nb_pixels:4d} pts, mean (col, row)=({col:.1f}, {row:.1f}), "
          f"std={stats.std_orientation:.2f}°, max misorientation={stats.max_angular_deviation:.2f}°")

In [ ]:
# clusters built with spatial connectivity are connected by construction.
# split_non_compact_clusters() is useful after clustering without spatial connectivity
# (max_gap=1: points touching by edge or corner are connected, max_gap=2 bridges 1-point gaps)
split_result = OC.split_non_compact_clusters(result, analyzer, max_gap=1, min_cluster_size=MIN_CLUSTER_SIZE)
print(split_result)
print('cluster 0 compact:', OC.is_cluster_compact(result.get_cluster(0), analyzer))

# MAPS of clusters

## all clusters

At map points where several grains are indexed (g0, g1, g2 ...), the grain with the largest number of indexed spots is shown. Black lines: boundaries between clusters. Hover the map to read image index and cluster id.

In [ ]:
#fig.clear('All')

fig, ax = OC.plot_cluster_map(result, analyzer, min_size=MIN_CLUSTER_SIZE, show_ids=True, show_boundaries=True)

## each individual cluster

Each cluster is colored by the misorientation (deg) of its points to the cluster reference matrix (red star: point closest to the cluster barycenter). Gray: other measured points.

In [ ]:
# all clusters (largest first). vmax=None: each panel has its own color scale, vmax=1: common scale
# live=False (default): shown as a PNG image (light for jupyterlab); live=True: interactive plot (zoom, hover values)
fig = OC.plot_clusters_gallery(result, analyzer, min_size=30, max_clusters=8, nb_cols=4, vmax=None, zoom=False, live=False)

In [ ]:
# browse clusters one by one: click in the list, then use the up/down arrow keys (or the prev/next buttons)
# property menu: GROD, KAM, Euler angles, strain (sample or crystal frame), lattice parameters, nb of spots, pixel deviation
# each value is read from the .fit file (and grain block) of the matrix of the cluster at each point
# 'crystal axes of cluster ref.': crystal-frame quantities expressed in the same cubic setting for all points
# 'live plot' checkbox: unchecked (default) maps shown as PNG images (fast, light); checked: interactive canvas
# the reference UB matrix of the selected cluster is shown as a copyable list (click in the box, Ctrl+A, Ctrl+C)
OC.visualize_cluster_interactive(result, analyzer, min_size=MIN_CLUSTER_SIZE, zoom=False, properties=True)

In [ ]:
fig.clear('All')

fig, ax = OC.plot_single_cluster(result, analyzer, cluster_id=2, zoom=False)

## properties of a cluster (strain, lattice parameters, Euler angles)

Values are read, for each point, from the `_g#.fit` file (and grain block) of the matrix belonging to the cluster at this point.

Cubic materials: crystal-frame quantities (crystal strain, lattice parameters, Euler angles) are brought to the axes of the cluster reference matrix (`aligned=True`), otherwise they may jump from point to point between equivalent cubic settings:
- crystal-frame strain: recomputed from the sample-frame strain (setting independent) in the axes of the aligned UB
- lattice parameters: written values transformed to the aligned axes (a keeps its fixed value of the deviatoric refinement)
- Euler angles: LaueTools convention (`orientations.calc_Euler_angles`) of the aligned UB

Properties: `OC.PROPERTY_GROUPS`

In [ ]:
print(OC.PROPERTY_GROUPS)
fig.clear('All')

fig, ax = OC.plot_cluster_property(result, analyzer, cluster_id=1, prop='ezz_sample', aligned=True,
                                   zoom=False, symmetric=False, percentiles=(2, 98), show_envelope=True)

# values as 2D array (NaN outside the cluster, strain in absolute units)
exz_c0 = OC.cluster_property_map(result, analyzer, cluster_id=0, prop='exz', aligned=True)

In [ ]:
# whole map: property of the primary grain of each point, crystal-frame quantities in the axes of the reference of its cluster
settings = OC.check_crystal_frame_settings(analyzer)
prop = 'b'
bmap = OC.property_map(result, analyzer, prop, aligned=True)
fig, ax = OC.plot_kam(bmap, result, analyzer, cmap='viridis',
                      vmin=np.nanpercentile(bmap, 1), vmax=np.nanpercentile(bmap, 99),
                      title=OC.property_label(prop) + ' (crystal axes of cluster references)')

# KAM and GROD

**Kernel Average Misorientation**: for each map point, mean misorientation between its orientation and those of its neighbours (`connectivity='8'`: 8 first neighbours, or an integer d for all points within distance d). Neighbours with misorientation > `max_angle` (grain boundaries) or belonging to another cluster are ignored. KAM reveals local lattice curvature, i.e. geometrically necessary dislocations.

**GROD** (Grain Reference Orientation Deviation): misorientation of each point to the reference orientation of its cluster, i.e. long-range orientation gradients inside grains.

In [ ]:
kam = OC.compute_kam(analyzer, result, connectivity='8', max_angle=2.0)
print(f'KAM mean {np.nanmean(kam):.3f}°, median {np.nanmedian(kam):.3f}°')
fig.clear('All')

fig, ax = OC.plot_kam(kam, result, analyzer, vmax=None, show_boundaries=False)

In [ ]:
grod = OC.compute_grod_map(result, analyzer)
fig.clear('All')

fig, ax = OC.plot_kam(grod, result, analyzer, cmap='viridis', show_boundaries=True,
                      title='GROD: misorientation to cluster reference (deg)')

## GROD and KAM of a single cluster

Only the orientations of the selected cluster are used, at all its points (also where it is not the primary grain of the point). Red line: envelope of the grain.

GROD reference: `None` = point closest to the cluster barycenter (red star), `'mean'` = mean orientation of the cluster, an image index of the cluster, or any UB matrix.

In [ ]:
cluster_id = 2

fig.clear('All')
fig = OC.plot_cluster_grod_kam(result, analyzer, cluster_id=cluster_id,
                               reference=None,          # None, 'mean', image index, or UB matrix
                               zoom=False, show_envelope=True,
                               kam_max_angle=2.0, kam_connectivity='8',
                               vmax_grod=None, vmax_kam=None)

# maps as arrays (NaN outside the cluster)
grod_c = OC.compute_grod_map(result, analyzer, cluster_id=cluster_id, reference='mean')
kam_c = OC.compute_kam(analyzer, result, cluster_id=cluster_id)
plt.show()

In [ ]:
# browse clusters (click in the list, then up/down arrow keys) and choose the GROD reference
OC.cluster_grod_interactive(result, analyzer, min_size=MIN_CLUSTER_SIZE, zoom=False, show_envelope=True)

# CLUSTER boundaries and envelopes

Even large clusters are sparse (points of other grains in between, several grains per image). The **envelope** of a grain is the region it occupies:

- `method='closing'` (default): morphological closing with a disk of `radius` map steps + hole filling. Bridges gaps up to ~2*radius and follows concave shapes. Envelope pieces with less than 3 points (stray points) are removed
- `method='hull'`: convex hull, i.e. extreme limits of the grain (overestimates concave grains)
- `method='bbox'`: bounding box

`fill` = fraction of the envelope actually occupied by the cluster points.

In [ ]:
# main clusters superimposed: envelope outline + light filling + points (slightly shifted per cluster
# so that grains superimposed at the same point remain visible)

fig.clear('All')
fig, ax, envelopes = OC.plot_cluster_envelopes(result, analyzer, min_size=100, max_clusters=10,
                                               method='closing', radius=2.0, show_points=True, fill_alpha=0.15)
for cid, env in envelopes.items():
    rmin, rmax, cmin, cmax = env['bbox']
    print(f"cluster {cid:3d}: {env['nb_points']:4d} pts, envelope area {env['area']:4d}, fill {100 * env['fill_fraction']:3.0f}%, "
          f"rows {rmin}-{rmax}, cols {cmin}-{cmax}, extent {env['extent']:.1f} steps, stray pts {env['nb_outliers']}")

In [ ]:
fig.clear('all')
# extreme limits (convex hulls) on top of the KAM map
fig, ax, hulls = OC.plot_cluster_envelopes(result, analyzer, min_size=100, max_clusters=15, method='hull',
                                           show_points=False, fill_alpha=0.05, background=kam)
plt.show()

## misorientation between neighbouring clusters

Pairs of clusters whose envelopes touch (`contact`) or overlap (`overlap`, grains superimposed along the beam or interpenetrating), misorientation between mean orientations and rotation axis (crystal frame of A). For cubic crystals, **Σ3 twins** (60° about <111>, Brandon criterion 8.66°) are identified.

In [ ]:
pairs = OC.cluster_pairs_table(result, analyzer, cluster_ids=list(envelopes), envelopes=envelopes,
                               low_angle=5.0, high_angle=15.0)

## boundary map

Boundaries between neighbouring points whose primary grains belong to different clusters, colored by misorientation class. Where several grains are superimposed, the primary grain (most indexed spots) may switch from point to point, which also produces boundary segments.

In [ ]:
fig.clear('All')

fig, ax, bnd = OC.plot_boundary_map(result, analyzer, low_angle=5.0, high_angle=15.0, background='clusters')

plt.show()

# [OPTION] REFINE AGAIN each grain with its own orientation

Each grain (cluster) found above is refined again, at all its map points, starting from its own UB matrices (as `usepreviousUB` + `skipindexing` of notebook 1, but with **image-dependent** matrices), with new refinement parameters: energy range (nb of spots of the model), tolerance angles, minimum matching rate ... No indexing from scratch: fast, and no wrong grain found at noisy points.

- **starting matrices** (`START`): `'local'`: UB matrix of the grain at this point (or at the nearest point of the grain), then the cluster reference matrix if the first one does not match; `'reference'`: cluster reference matrix only
- **map points** (`REGION`, `DILATE`): `'points'` of the cluster; `'envelope'`: + holes inside the grain envelope (grain missed, or indexed as another grain index); `'box'`: all points of a box centred on the cluster barycenter (`BOX_HALFSIZE` map steps); + `DILATE` map steps around (grain borders)
- **execution**: in this jupyter session (`EXECUTION = 'local'`) or as a job on the ESRF cluster (`'slurm'`, as in notebook 1)
- **results**: 1 folder per grain `GRAINS_FOLDER/grain_###/prefix####_g0.fit` (1 grain per .fit file: each grain is refined independently of the other grains of the image). Settings are saved in `GRAINS_FOLDER/grain_refinement.yaml`
- maps (strain, lattice parameters ...) of a refined grain: `2_postprocess_maps.ipynb` with `FIT_FOLDER = GRAINS_FOLDER/grain_###`

The same grain is often indexed several times at a map point (g0, g1 ... with different subsets of spots). After refinement there is one matrix per point, so sub-grain boundaries bridged by these duplicates may appear when the refined grain is segmented again with the same `THRESHOLD`. Compare KAM before / after (cell below) to judge whether new parameters reduce the orientation noise.

In [ ]:
GRAIN_IDS = None          # None: all clusters with at least GRAIN_MIN_SIZE matrices (largest first), or e.g. [0, 2, 5]
GRAIN_MIN_SIZE = 100
REGION = 'envelope'       # 'points': points of the cluster; 'envelope': + holes inside the grain envelope; 'box': box around barycenter
BOX_HALFSIZE = (10, 10)   # REGION = 'box': half size (fast axis, slow axis) in map steps
DILATE = 1                # + map points within DILATE map steps (grain borders)
START = 'local'           # 'local': UB of the grain at this point (or nearest point), then cluster reference; 'reference': reference only
GRAINS_FOLDER = Path(f'{fit_folder}_grains')   # .fit files in GRAINS_FOLDER/grain_###/

grain_tasks = IB.grain_refinement_tasks(result, analyzer, cluster_ids=GRAIN_IDS, min_size=GRAIN_MIN_SIZE,
                                        region=REGION, dilate=DILATE, box_halfsize=BOX_HALFSIZE, start=START,
                                        align=True)   # same hkl for a reflection at all points of a grain
print(f'{len(grain_tasks)} grains, {sum(len(t) for t in grain_tasks.values())} images to refine -> {GRAINS_FOLDER}')

In [ ]:
RUN_GRAIN_REFINEMENT = False   # True: refine (overwrites the .fit files of GRAINS_FOLDER/grain_###)
EXECUTION_GRAINS = 'local'     # 'local': this jupyter session; 'slurm': job on the ESRF cluster
NB_CPUS_GRAINS = None          # local: None = all cpus available
SLURM_GRAINS = dict(machine='magnifix', nb_cpus=192, time='0:30:00', mem_per_cpu='2100M', nchunks=1)
SUBMIT = True                  # slurm: True = sbatch now (jupyter-slurm session), False = only prepare the job

# refinement parameters: those of the configuration file, changed here
params_grains = IB.params_from_config(cfg)
params_grains.e_max = cfg['indexing']['e_max']                                # keV: larger -> more spots in the model
params_grains.dict_indexrefine['list matching tol angles'] = [0.2, 0.15, 0.1]  # deg, tolerance of each refinement step
params_grains.dict_indexrefine['MinimumMatchingRate'] = 5                      # %, min matching rate to accept a starting UB
params_grains.MatchingRate_List = [2] * len(params_grains.dict_indexrefine['list matching tol angles'])
# params_grains.MAXNBSPOTS = 300                                               # nb of (most intense) spots read in .cor files

grain_results, grain_job = None, None
if RUN_GRAIN_REFINEMENT and EXECUTION_GRAINS == 'local':
    grain_results = IB.refine_grains(cfg, grain_tasks, params_grains, GRAINS_FOLDER, nb_cpus=NB_CPUS_GRAINS, result=result)
elif RUN_GRAIN_REFINEMENT and EXECUTION_GRAINS == 'slurm':
    grain_job = IB.prepare_slurm_grain_job(cfg, grain_tasks, params_grains, GRAINS_FOLDER, result=result, **SLURM_GRAINS)
    if SUBMIT:
        IB.submit_slurm_job(grain_job)

In [ ]:
# slurm: job state and last lines of the log(s) (run again to update), then results when finished
# after a restart of the kernel: grain_job = '<job folder>'
if grain_job is not None:
    IB.slurm_job_status(grain_job)
    grain_results = IB.load_slurm_grain_results(grain_job)

In [ ]:
# a grain before / after refinement: KAM inside the grain, and segmentation of the refined grain
cluster_id = 0
grain_dir = IB.grain_folder(GRAINS_FOLDER, cluster_id)
if grain_dir.is_dir():
    lay = IB.fitfiles_layout(cfg)
    result_g, analyzer_g = OC.analyze_ub_matrices(input_dir=grain_dir, threshold=THRESHOLD, mapdimension=invmapdims,
                                                  prefix=scan['prefix'], symmetry=result.symmetry, check_symmetry=False,
                                                  nbfiles_per_folder=lay['fit_nbfiles_per_folder'],
                                                  subfolder_prefix=lay['subfolder_prefix'])
    before = result.get_cluster(cluster_id)
    print(f'before: {before.size} matrices at {before.nb_pixels} points; after: {len(analyzer_g.matrices)} matrices '
          f'-> {len(result_g.clusters)} clusters at threshold {THRESHOLD} deg, sizes {[c.size for c in result_g.clusters[:5]]}')
    kam_before = OC.compute_kam(analyzer, result, cluster_id=cluster_id, max_angle=2.0)
    kam_after = OC.compute_kam(analyzer_g, None, max_angle=2.0)   # all matrices of grain_dir belong to the grain
    vmax = np.nanpercentile(kam_before, 98)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, kam_, res_, an_, title in ((axes[0], kam_before, result, analyzer, 'before'),
                                       (axes[1], kam_after, result_g, analyzer_g, 'after refinement')):
        OC.plot_kam(kam_, res_, an_, vmax=vmax, ax=ax, show_boundaries=False,
                    title=f'grain {cluster_id} KAM {title}: mean {np.nanmean(kam_):.3f} deg')
    plt.show()
else:
    print(f'{grain_dir} does not exist: run the grain refinement (RUN_GRAIN_REFINEMENT = True)')

## [OPTION] 2nd refinement with a FIXED SET of reliable reflections per grain

Strain and lattice parameters depend on the spots used. Here each grain is refined again with the **same reflections** at all its points, keeping only reliable spots, from the results of the refinement above (`GRAINS_FOLDER`):

1. **reliable spots** (peak search columns of the .cor files): `max_dev` = max distance (pixel) between fitted spot center and initial position (`Xdev`, `Ydev`: large for asymmetric or multi-component spots), `max_elongation` = max ratio of the 2 fwhm, `max_fwhm`, `min_isolation` = min distance (pixel) to the nearest other spot of the image (spots of other grains, spurious spots). NB: peak search does not keep spots closer than ~10 pixels
2. **reference reflections** of the grain: hkl indexed AND reliable in at least `min_fraction` of the points of the grain (`nb_spots`: keep only the most frequent ones)
3. for each point: .cor file with only the reference reflections reliable at this point (points with less than `min_spots` are skipped; `require_all=True`: only points where all reference reflections are reliable), then refinement starting from the UB matrix of step 1, tolerance angles of `params_fixed`

A strictly identical set at all points is rarely possible (spots vary from point to point): the table below gives, for a few criteria, the nb of reference reflections, of points kept and of spots per point. Fewer spots give larger statistical uncertainties: compare the strain noise of both refinements (last cell) before choosing.

In [ ]:
cluster_id = 0
spots_g, UBs_g = IB.read_grain_spots(cfg, IB.grain_folder(GRAINS_FOLDER, cluster_id))
print(spots_g[['dev', 'elongation', 'isolation', 'pixDev', 'peak_fwaxmaj']].describe().round(2))
n_img = spots_g['image'].nunique()
for max_dev in (1.0, 2.0, None):
    for max_elongation in (2.0, None):
        good = IB.spot_quality(spots_g, max_dev=max_dev, max_elongation=max_elongation, min_isolation=15)
        for min_fraction in (0.3, 0.5, 0.7):
            table = IB.reference_reflections(spots_g, good, min_fraction=min_fraction)
            reference = set(table.index[table['selected']])
            keep = good & np.array([hkl in reference for hkl in zip(spots_g['h'], spots_g['k'], spots_g['l'])])
            per_image = spots_g[keep].groupby('image').size().reindex(sorted(spots_g['image'].unique()), fill_value=0)
            print(f'max_dev {max_dev}, max_elongation {max_elongation}, min_fraction {min_fraction}: {len(reference):3d} reflections, '
                  f'{100 * good.mean():3.0f}% reliable spots, points with >= 8 spots {int((per_image >= 8).sum())}/{n_img}, '
                  f'median {per_image.median():.0f} spots/point')

In [ ]:
RUN_FIXED_SPOTS = False
FIXED_FOLDER = Path(f'{GRAINS_FOLDER}_fixedspots')    # .fit files of the 2nd refinement: FIXED_FOLDER/grain_###/

if RUN_FIXED_SPOTS:
    fixed_tasks, fixed_cor_folders, reference_tables = IB.grain_spots_selection(
        cfg, GRAINS_FOLDER, cluster_ids=list(grain_tasks),
        max_dev=2.0, max_elongation=2.0, max_fwhm=None, min_isolation=15,   # reliable spots
        min_fraction=0.3, nb_spots=None,                                     # reference reflections of each grain
        min_spots=8, require_all=False)          # .cor files written in GRAINS_FOLDER/selected_corfiles/grain_###/
    display(reference_tables[list(reference_tables)[0]].round(2))

    params_fixed = IB.params_from_config(cfg)
    params_fixed.dict_indexrefine['list matching tol angles'] = [0.2, 0.1, 0.05]   # deg
    params_fixed.dict_indexrefine['MinimumMatchingRate'] = 1    # %: few experimental spots -> low matching rates
    params_fixed.MatchingRate_List = [1] * len(params_fixed.dict_indexrefine['list matching tol angles'])
    if EXECUTION_GRAINS == 'local':
        fixed_results = IB.refine_grains(cfg, fixed_tasks, params_fixed, FIXED_FOLDER, cor_folders=fixed_cor_folders,
                                         nb_cpus=NB_CPUS_GRAINS, result=result)
    else:
        grain_job = IB.prepare_slurm_grain_job(cfg, fixed_tasks, params_fixed, FIXED_FOLDER, cor_folders=fixed_cor_folders,
                                               result=result, **SLURM_GRAINS)
        if SUBMIT:
            IB.submit_slurm_job(grain_job)   # follow it with the slurm cell above

In [ ]:
# strain noise: rms of the differences between neighbouring points (same points in both refinements), 1e-4
def strain_map(folder):
    an = OC.UBMatrixAnalyzer(input_dir=folder, prefix=scan['prefix'], mapdimension=invmapdims)
    an.load_matrices()
    smap = np.full(tuple(invmapdims) + (6,), np.nan)
    for ub in an.matrices:
        e = ub.props.get('strain_sample')
        if e is not None:
            e = np.asarray(e)
            smap[divmod(ub.image_index, invmapdims[1])] = [e[0, 0], e[1, 1], e[2, 2], e[0, 1], e[0, 2], e[1, 2]]
    return smap

cluster_id = 0
if IB.grain_folder(FIXED_FOLDER, cluster_id).is_dir():
    s1, s2 = strain_map(IB.grain_folder(GRAINS_FOLDER, cluster_id)), strain_map(IB.grain_folder(FIXED_FOLDER, cluster_id))
    both = ~np.isnan(s1[..., 0]) & ~np.isnan(s2[..., 0])
    for name, smap in (('all spots ', s1), ('fixed set ', s2)):
        diffs = [(smap[:, 1:] - smap[:, :-1])[both[:, 1:] & both[:, :-1]], (smap[1:] - smap[:-1])[both[1:] & both[:-1]]]
        rms = 1e4 * np.sqrt(np.mean(np.concatenate(diffs) ** 2, axis=0))
        print(f'{name} neighbour rms xx yy zz xy xz yz: {rms.round(2)}, mean: {(1e4 * smap[both].mean(axis=0)).round(2)}  ({both.sum()} points)')